# Early Sepsis Prediction Experiment — MIMIC-III Sepsis-3

**Iteration 3 — Kaggle experiment notebook**

This notebook compares RNN, LSTM, CNN-Transformer, and LSTM-Transformer using the processed 12-hour × 22-feature temporal tensors in this repository. The current processed data contain 4h and 8h prediction horizons; the 12h horizon is unavailable because no positive cases satisfy the current onset/window eligibility design.

> Important: the current 4h/8h tensors have very few Sepsis-3 positive cases. Treat the results as feasibility evidence unless the cohort issue is corrected or explicitly accepted as a study limitation.

## Kaggle setup
Attach the repository data as a private Kaggle Dataset or clone the private repository manually. Use a GPU accelerator. The notebook searches common Kaggle paths for `data/dataset_manifest.json`.

In [ ]:
MODE='pilot'  # pilot=3 epochs; final=20 epochs + 5-fold CV
SEED=42; HORIZONS=[4,8]
MODELS=['rnn','lstm','cnn_transformer','lstm_transformer']
PILOT_EPOCHS=3; FINAL_EPOCHS=20; BATCH_SIZE=128
LR=1e-3; WEIGHT_DECAY=1e-4; HIDDEN=64; DROPOUT=0.2; PATIENCE=5
TEST_SIZE=.30; VAL_SIZE=.20; CV_FOLDS=5
RUN_BOOTSTRAP=True; BOOTSTRAP_SAMPLES=500; RUN_LOCAL_SHAP=False

In [ ]:
from pathlib import Path
from copy import deepcopy
import json,math,random,pickle,warnings
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.metrics import *
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
warnings.filterwarnings('ignore'); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('Device:',DEVICE)

## 1. Load processed data and audit the experiment inputs

In [ ]:
def locate_data():
    c=[Path.cwd()/'data',Path.cwd().parent/'data',Path('/kaggle/working/sepsis_test/data'),Path('/kaggle/input/sepsis-test/data')]
    root=Path('/kaggle/input')
    if root.exists():
        for p in root.glob('*'): c += [p/'data',p]
    for p in c:
        if (p/'dataset_manifest.json').exists(): return p.resolve()
    raise FileNotFoundError('Attach the repository data to Kaggle.')
DATA_DIR=locate_data(); PROCESSED=DATA_DIR/'processed'
OUT=Path('/kaggle/working/sepsis_experiment'); OUT.mkdir(parents=True,exist_ok=True)
manifest=json.loads((DATA_DIR/'dataset_manifest.json').read_text()); data={}
for h in HORIZONS:
    z=np.load(PROCESSED/f'timeseries_before_onset_{h}h.npz',allow_pickle=False); data[h]={k:z[k] for k in z.files}; z.close()
rows=[]
for h,d in data.items():
    x=d['x']; y=d['y'].astype(int); allmiss=np.isnan(x).all((1,2))
    rows.append({'horizon':h,'shape':str(x.shape),'patients':len(y),'sepsis':int(y.sum()),'controls':int((y==0).sum()),'prevalence_%':100*y.mean(),'missing_%':100*np.isnan(x).mean(),'all_missing':int(allmiss.sum())})
display(pd.DataFrame(rows).round(3)); print('Label:',manifest.get('label_definition')); print('12h:',manifest.get('omitted_12h'))

## 2. Remove completely unobserved windows and split the data

The report uses a stratified 70% development / 30% held-out test split. The development set is further divided into training and validation data. Patient IDs are checked to prevent overlap.

In [ ]:
clean={}; splits={}
for h,d in data.items():
    x=d['x'].astype('float32'); y=d['y'].astype(int); keep=~np.isnan(x).all((1,2))
    dd={};
    for k,v in d.items(): dd[k]=v[keep] if isinstance(v,np.ndarray) and v.ndim and v.shape[0]==len(y) else v
    clean[h]=dd; y=dd['y'].astype(int); sid=dd['subject_id']; idx=np.arange(len(y))
    dev,test=train_test_split(idx,test_size=TEST_SIZE,random_state=SEED,stratify=y)
    tr,va=train_test_split(dev,test_size=VAL_SIZE,random_state=SEED,stratify=y[dev])
    assert set(sid[tr]).isdisjoint(set(sid[va])) and set(sid[tr]).isdisjoint(set(sid[test])) and set(sid[va]).isdisjoint(set(sid[test]))
    splits[h]={'train':tr,'val':va,'dev':dev,'test':test}
    print(h,'h:',[(n,len(i),int(y[i].sum())) for n,i in [('train',tr),('val',va),('test',test)]])

## 3. Leakage-safe imputation and scaling
Remaining NaNs are mean-imputed and standardised using **training data only**.

$$z_{i,t,f}=\frac{\tilde{x}_{i,t,f}-\mu_f^{train}}{\sigma_f^{train}}$$

In [ ]:
def fit_prep(x):
    f=x.shape[2]; imp=SimpleImputer(strategy='mean',keep_empty_features=True); sc=StandardScaler(); flat=x.reshape(-1,f); sc.fit(imp.fit_transform(flat)); return imp,sc
def tx(x,imp,sc):
    n,t,f=x.shape; return sc.transform(imp.transform(x.reshape(-1,f))).reshape(n,t,f).astype('float32')
prep={}; proc={}
for h,d in clean.items():
    x=d['x'].astype('float32'); y=d['y'].astype(int); s=splits[h]; imp,sc=fit_prep(x[s['train']]); prep[h]=(imp,sc)
    proc[h]={'x_train':tx(x[s['train']],imp,sc),'y_train':y[s['train']],'x_val':tx(x[s['val']],imp,sc),'y_val':y[s['val']],'x_test':tx(x[s['test']],imp,sc),'y_test':y[s['test']]}
    pos=int(proc[h]['y_train'].sum()); neg=int((proc[h]['y_train']==0).sum()); print(h,'h pos_weight=',neg/max(pos,1))

## 4. Evaluation functions and prior baseline
Required report metrics: Accuracy, Precision, Recall, F1, AUROC, AUPR. Supplementary diagnostics: Specificity, Balanced Accuracy, MCC, and Brier score.

In [ ]:
def metric_pack(y,s,thr=.5):
    y=np.asarray(y).astype(int); s=np.asarray(s,float); p=(s>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'accuracy':accuracy_score(y,p),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'specificity':tn/(tn+fp) if tn+fp else np.nan,'f1':f1_score(y,p,zero_division=0),'balanced_accuracy':balanced_accuracy_score(y,p),'mcc':matthews_corrcoef(y,p) if len(np.unique(p))>1 else 0,'brier':brier_score_loss(y,s),'auroc':roc_auc_score(y,s),'aupr':average_precision_score(y,s)}
base=[]
for h,p in proc.items():
    m=DummyClassifier(strategy='prior').fit(np.zeros((len(p['y_train']),1)),p['y_train']); s=m.predict_proba(np.zeros((len(p['y_test']),1)))[:,1]; r=metric_pack(p['y_test'],s); r.update({'horizon':h,'model':'prior_baseline'}); base.append(r)
baseline_df=pd.DataFrame(base); display(baseline_df.round(4))

## 5. Model architectures

Transformer attention uses:

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class PE(nn.Module):
    def __init__(self,d,max_len=100):
        super().__init__(); pos=torch.arange(max_len).float().unsqueeze(1); div=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000)/d)); pe=torch.zeros(max_len,d); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]]); self.register_buffer('pe',pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]
class RNNM(nn.Module):
    def __init__(self,f): super().__init__(); self.r=nn.RNN(f,HIDDEN,batch_first=True); self.h=nn.Linear(HIDDEN,1)
    def forward(self,x): _,h=self.r(x); return self.h(h[-1]).squeeze(-1)
class LSTMM(nn.Module):
    def __init__(self,f): super().__init__(); self.r=nn.LSTM(f,HIDDEN,batch_first=True); self.h=nn.Linear(HIDDEN,1)
    def forward(self,x): _,(h,_)=self.r(x); return self.h(h[-1]).squeeze(-1)
class CNNT(nn.Module):
    def __init__(self,f): super().__init__(); self.c=nn.Conv1d(f,HIDDEN,3,padding=1); self.p=PE(HIDDEN); layer=nn.TransformerEncoderLayer(HIDDEN,4,HIDDEN*4,DROPOUT,batch_first=True); self.e=nn.TransformerEncoder(layer,2); self.h=nn.Linear(HIDDEN,1)
    def forward(self,x): x=self.c(x.transpose(1,2)).transpose(1,2); x=self.e(self.p(x)); return self.h(x.mean(1)).squeeze(-1)
class LSTMT(nn.Module):
    def __init__(self,f): super().__init__(); self.l=nn.LSTM(f,HIDDEN,batch_first=True); self.p=PE(HIDDEN); layer=nn.TransformerEncoderLayer(HIDDEN,4,HIDDEN*4,DROPOUT,batch_first=True); self.e=nn.TransformerEncoder(layer,2); self.h=nn.Linear(HIDDEN,1)
    def forward(self,x): x,_=self.l(x); x=self.e(self.p(x)); return self.h(x.mean(1)).squeeze(-1)
def build(name,f): return {'rnn':RNNM,'lstm':LSTMM,'cnn_transformer':CNNT,'lstm_transformer':LSTMT}[name](f)

## 6. Training
Weighted BCE uses $w_+=N_{negative}/N_{positive}$. Training uses AdamW, gradient clipping, validation loss, and early stopping.

In [ ]:
def predict(m,x):
    m.eval(); o=[]; dl=DataLoader(TensorDataset(torch.tensor(x,dtype=torch.float32)),batch_size=BATCH_SIZE)
    with torch.no_grad():
        for (xb,) in dl: o.append(torch.sigmoid(m(xb.to(DEVICE))).cpu().numpy())
    return np.concatenate(o)
def train_one(name,xtr,ytr,xv,yv,epochs):
    m=build(name,xtr.shape[2]).to(DEVICE); pos=max(1,int(ytr.sum())); neg=max(1,int((ytr==0).sum())); w=neg/pos
    lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([w],dtype=torch.float32,device=DEVICE)); opt=torch.optim.AdamW(m.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    dl=DataLoader(TensorDataset(torch.tensor(xtr,dtype=torch.float32),torch.tensor(ytr,dtype=torch.float32)),batch_size=BATCH_SIZE,shuffle=True)
    best=None; bv=float('inf'); wait=0; hist=[]
    for ep in range(1,epochs+1):
        m.train(); ls=[]
        for xb,yb in dl:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=lossfn(m(xb),yb); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); ls.append(float(loss.detach().cpu()))
        m.eval();
        with torch.no_grad(): vl=float(lossfn(m(torch.tensor(xv,dtype=torch.float32,device=DEVICE)),torch.tensor(yv,dtype=torch.float32,device=DEVICE)).cpu())
        tl=float(np.mean(ls)); hist.append({'epoch':ep,'train_loss':tl,'val_loss':vl}); print(name,ep,tl,vl)
        if vl<bv-1e-6: bv=vl; best=deepcopy(m.state_dict()); wait=0
        else: wait+=1
        if wait>=PATIENCE: break
    if best is not None: m.load_state_dict(best)
    return m,pd.DataFrame(hist),w

## 7. Five-fold cross-validation on the development set

In [ ]:
def run_cv(h,name,d,dev,epochs):
    x=d['x'].astype('float32')[dev]; y=d['y'].astype(int)[dev]
    if np.bincount(y,minlength=2).min()<CV_FOLDS: print('CV skipped: too few positives'); return pd.DataFrame()
    rows=[]; skf=StratifiedKFold(CV_FOLDS,shuffle=True,random_state=SEED)
    for fold,(tr,va) in enumerate(skf.split(x,y),1):
        imp,sc=fit_prep(x[tr]); xt,xv=tx(x[tr],imp,sc),tx(x[va],imp,sc); m,_,_=train_one(name,xt,y[tr],xv,y[va],epochs); r=metric_pack(y[va],predict(m,xv)); r.update({'horizon':h,'model':name,'fold':fold,'validation_positives':int(y[va].sum())}); rows.append(r)
    return pd.DataFrame(rows)

## 8. Run the complete experiment

In [ ]:
epochs=PILOT_EPOCHS if MODE=='pilot' else FINAL_EPOCHS; trained={}; histories={}; rows=[]; preds=[]; cvs=[]
for h,p in proc.items():
    trained[h]={}; histories[h]={}
    for name in MODELS:
        print('\n',h,'h',name); m,hist,w=train_one(name,p['x_train'],p['y_train'],p['x_val'],p['y_val'],epochs); trained[h][name]=m; histories[h][name]=hist
        s=predict(m,p['x_test']); r=metric_pack(p['y_test'],s); r.update({'horizon':h,'model':name,'test_positives':int(p['y_test'].sum()),'positive_weight':w}); rows.append(r)
        idx=splits[h]['test']; d=clean[h]; preds.append(pd.DataFrame({'horizon':h,'model':name,'subject_id':d['subject_id'][idx],'icustay_id':d['icustay_id'][idx],'true_label':p['y_test'],'predicted_probability':s}))
        torch.save({'model_state':m.state_dict(),'model_name':name,'feature_names':d['feature_names'].astype(str).tolist()},OUT/f'model_{name}_{h}h.pt')
        with open(OUT/f'preprocess_{name}_{h}h.pkl','wb') as f: pickle.dump({'imputer':prep[h][0],'scaler':prep[h][1]},f)
        if MODE=='final':
            cv=run_cv(h,name,d,splits[h]['dev'],epochs);
            if len(cv): cvs.append(cv)
results=pd.DataFrame(rows); predictions=pd.concat(preds,ignore_index=True); results.to_csv(OUT/'model_metrics.csv',index=False); predictions.to_csv(OUT/'heldout_predictions.csv',index=False); display(results.round(4))
cv_df=pd.concat(cvs,ignore_index=True) if cvs else pd.DataFrame();
if len(cv_df): cv_df.to_csv(OUT/'cross_validation_metrics.csv',index=False)

## 9. Training curves, confusion matrices, ROC and PR curves

In [ ]:
for h,mods in trained.items():
    p=proc[h]
    for name,m in mods.items():
        hist=histories[h][name]; plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.train_loss,label='Train'); plt.plot(hist.epoch,hist.val_loss,label='Validation'); plt.title(f'{name} {h}h loss'); plt.legend(); plt.show()
        s=predict(m,p['x_test']); pred=(s>=.5).astype(int); ConfusionMatrixDisplay.from_predictions(p['y_test'],pred,display_labels=['Non-sepsis','Sepsis-3'],values_format='d'); plt.title(f'{name} {h}h'); plt.show()
    plt.figure(figsize=(6,5))
    for name,m in mods.items():
        s=predict(m,p['x_test']); fpr,tpr,_=roc_curve(p['y_test'],s); plt.plot(fpr,tpr,label=f'{name} {roc_auc_score(p["y_test"],s):.3f}')
    plt.plot([0,1],[0,1],'--'); plt.title(f'ROC {h}h'); plt.legend(); plt.show()
    plt.figure(figsize=(6,5)); prev=p['y_test'].mean()
    for name,m in mods.items():
        s=predict(m,p['x_test']); pr,re,_=precision_recall_curve(p['y_test'],s); plt.plot(re,pr,label=f'{name} {average_precision_score(p["y_test"],s):.3f}')
    plt.axhline(prev,ls='--',label=f'prevalence={prev:.3f}'); plt.title(f'PR {h}h'); plt.legend(); plt.show()

## 10. Cross-validation summary

In [ ]:
if len(cv_df):
    cols=['accuracy','precision','recall','specificity','f1','balanced_accuracy','mcc','auroc','aupr','brier']; summary=cv_df.groupby(['horizon','model'])[cols].agg(['mean','std']).reset_index(); display(summary); summary.to_csv(OUT/'cross_validation_summary.csv',index=False)
else: print('Run MODE=final for cross-validation.')

## 11. Result-analysis framework
For each horizon discuss: test positive count; performance versus baseline; whether positive patients were detected; whether accuracy is misleading; AUPR relative to prevalence; CV stability; 4h vs 8h performance; and why direct comparison with Tang et al. must consider dataset, cohort, onset definition, missingness, and class distribution.

### Required limitations
- very small post-filter positive sample;
- no 12h positive cohort under the current design;
- feature-dependent missingness;
- retrospective MIMIC data;
- no external validation;
- pseudo-onset assumptions for controls;
- explainability is not causal evidence;
- research prototype, not a clinical decision tool.

## Final checklist
Before reporting final results: run the EDA notebook; switch to `MODE='final'`; use a Kaggle GPU; train all four models; run 5-fold CV; report Accuracy, Precision, Recall, F1, AUROC and AUPR; include confusion matrices, ROC and PR curves; state the number of positive test patients; compare with Tang et al. critically; and disclose the small-positive-sample limitation.